# Incremental Structure-from-Motion

Pipeline logic lives in the `sfm/` package. This notebook loads data and runs each stage.

In [1]:
import importlib
import sfm
importlib.reload(sfm)

for _name in (
    "sfm.io",
    "sfm.geometry",
    "sfm.matching",
    "sfm.matching_superpoint",
    "sfm.tracks",
    "sfm.mapping",
    "sfm.ba",
    "sfm.viz",
    "sfm.colmap_export",
    "sfm.rng",
):
    importlib.reload(importlib.import_module(_name))
importlib.reload(sfm)

from sfm import (
    diagnose_map_quality,
    export_colmap_model,
    extract_features_and_matches,
    extract_features_and_matches_superpoint,
    find_best_seed,
    initialize_seed_map,
    load_all_intrinsics,
    load_data,
    build_global_tracks,
    open_in_colmap_gui,
    plot_3d_map,
    retriangulate_tracks,
    run_bundle_adjustment,
    run_pnp_mapping,
    set_random_seed,
)

# Deterministic F-RANSAC / PnP-RANSAC / NumPy draws across runs
set_random_seed(0)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Random seed set to 0 (NumPy, OpenCV; OpenCV threads=1)


## 1. Load images and intrinsics

In [2]:
image_dir = "buddha_images/"
Images = load_data(image_dir)
num_images = len(Images)
print(f"Loaded {num_images} images from {image_dir}")

all_intrinsics = load_all_intrinsics("cameras.txt")

Loaded 24 images from buddha_images/
Camera ID: 1
Focal Length: 1700.5960173760209
Principal Point: (540.0, 960.0)
Radial Distortion (k1): 0.04460722023943839
Intrinsic Matrix K:
[[1.70059602e+03 0.00000000e+00 5.40000000e+02]
 [0.00000000e+00 1.70059602e+03 9.60000000e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]

Camera ID: 2
Focal Length: 1696.9933605815868
Principal Point: (540.0, 960.0)
Radial Distortion (k1): 0.04684414515917654
Intrinsic Matrix K:
[[1.69699336e+03 0.00000000e+00 5.40000000e+02]
 [0.00000000e+00 1.69699336e+03 9.60000000e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]

Camera ID: 3
Focal Length: 1707.438317595852
Principal Point: (540.0, 960.0)
Radial Distortion (k1): 0.04551770441526508
Intrinsic Matrix K:
[[1.70743832e+03 0.00000000e+00 5.40000000e+02]
 [0.00000000e+00 1.70743832e+03 9.60000000e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]

Camera ID: 4
Focal Length: 1707.3555769968862
Principal Point: (540.0, 960.0)
Radial Distortion (k1)

## 2. Feature matching and global tracks

In [3]:
# SIFT (default): usually cleaner 3D structure on this dataset.
# SuperPoint/LightGlue: denser points, but often weaker structure — optional below.
keypoints, descriptors, matches = extract_features_and_matches(
    Images,
    nfeatures=8000,
    window_size=8,
    ratio_thresh=0.8,
    min_inliers=15,
    ransac_iterations=2000,
    contrast_threshold=0.02,
)
# keypoints, descriptors, matches = extract_features_and_matches_superpoint(
#     Images, max_num_keypoints=4096, window_size=8, min_inliers=15,
# )

tracks, img_kp_to_track = build_global_tracks(num_images, keypoints, matches)
avg_kps = sum(len(k) for k in keypoints) / max(len(keypoints), 1)
print(f"Keypoints/image (avg): {avg_kps:.0f}")
print(f"Verified pairs: {len(matches)}, multi-view tracks: {len(tracks)}")

Random seed set to 0 (NumPy, OpenCV; OpenCV threads=1)
Verified pair (0, 1) with 959 inliers.
Verified pair (0, 2) with 345 inliers.
Verified pair (0, 3) with 280 inliers.
Verified pair (0, 4) with 89 inliers.
Verified pair (0, 5) with 87 inliers.
Verified pair (0, 6) with 23 inliers.
Verified pair (0, 7) with 27 inliers.
Verified pair (0, 8) with 17 inliers.
Verified pair (1, 2) with 454 inliers.
Verified pair (1, 3) with 339 inliers.
Verified pair (1, 4) with 149 inliers.
Verified pair (1, 5) with 81 inliers.
Verified pair (1, 6) with 47 inliers.
Verified pair (1, 7) with 44 inliers.
Verified pair (1, 8) with 17 inliers.
Verified pair (1, 9) with 34 inliers.
Verified pair (2, 3) with 421 inliers.
Verified pair (2, 4) with 157 inliers.
Verified pair (2, 5) with 139 inliers.
Verified pair (2, 6) with 47 inliers.
Verified pair (2, 7) with 33 inliers.
Verified pair (2, 8) with 20 inliers.
Verified pair (2, 9) with 24 inliers.
Verified pair (2, 10) with 16 inliers.
Verified pair (3, 4) wi

## 3. Seed pair selection and initial triangulation

In [4]:
# Optional: search for a high-parallax seed (for inspection)
best_pair, seed_pose = find_best_seed(matches, all_intrinsics, keypoints)
print(f"find_best_seed suggests: {best_pair}")

# Use the proven seed pair from the original pipeline
seed_i, seed_j = 15, 16
map_3d, camera_poses, R_seed, t_seed = initialize_seed_map(
    seed_i, seed_j, matches, keypoints, img_kp_to_track, all_intrinsics
)

Cheirality Check: Found valid pose with 815 points in front of cameras.
Candidate (0, 1): 0.04° parallax - TOO LOW (Narrow Baseline)

Cheirality Check: Found valid pose with 453 points in front of cameras.
Candidate (1, 2): 0.22° parallax - TOO LOW (Narrow Baseline)

Cheirality Check: Found valid pose with 420 points in front of cameras.
Candidate (2, 3): 0.96° parallax - TOO LOW (Narrow Baseline)

Cheirality Check: Found valid pose with 353 points in front of cameras.
Candidate (17, 18): 353 inliers, 2.95° parallax - VALID

Cheirality Check: Found valid pose with 347 points in front of cameras.
Candidate (19, 20): 347 inliers, 2.11° parallax - VALID

Cheirality Check: Found valid pose with 345 points in front of cameras.
Candidate (0, 2): 0.45° parallax - TOO LOW (Narrow Baseline)

Cheirality Check: Found valid pose with 339 points in front of cameras.
Candidate (1, 3): 0.92° parallax - TOO LOW (Narrow Baseline)

Cheirality Check: Found valid pose with 333 points in front of cameras.


## 4. Incremental PnP registration (bi-directional)

In [5]:
forward_indices = range(seed_j + 1, len(keypoints))
reverse_indices = range(seed_i - 1, -1, -1)

print("Starting Forward Pass")
run_pnp_mapping(
    forward_indices,
    map_3d,
    camera_poses,
    img_kp_to_track,
    matches,
    keypoints,
    all_intrinsics,
)

print("\nStarting Reverse Pass")
run_pnp_mapping(
    reverse_indices,
    map_3d,
    camera_poses,
    img_kp_to_track,
    matches,
    keypoints,
    all_intrinsics,
)

print(f"\nMap after PnP: {len(map_3d)} points")
retriangulate_tracks(
    map_3d,
    camera_poses,
    tracks,
    keypoints,
    all_intrinsics,
)

Starting Forward Pass
Registered 17: 97 inliers, added 313 points.
Registered 18: 204 inliers, added 327 points.
Registered 19: 259 inliers, added 242 points.
Registered 20: 323 inliers, added 293 points.
Registered 21: 334 inliers, added 289 points.
Registered 22: 292 inliers, added 199 points.
Registered 23: 328 inliers, added 232 points.

Starting Reverse Pass
Registered 14: 242 inliers, added 202 points.
Registered 13: 112 inliers, added 90 points.
Registered 12: 95 inliers, added 131 points.
Registered 11: 136 inliers, added 231 points.
Registered 10: 171 inliers, added 170 points.
Registered 9: 163 inliers, added 154 points.
Registered 8: 128 inliers, added 165 points.
Registered 7: 82 inliers, added 150 points.
Registered 6: 120 inliers, added 240 points.
Registered 5: 161 inliers, added 249 points.
Registered 4: 143 inliers, added 139 points.
Registered 3: 128 inliers, added 159 points.
Registered 2: 185 inliers, added 414 points.
Registered 1: 368 inliers, added 333 points.
Re

7

## 5. Pre-BA diagnostics and visualization

In [7]:
diagnose_map_quality(map_3d, camera_poses, img_kp_to_track, keypoints)

plot_3d_map(
    map_3d,
    camera_poses,
    all_intrinsics,
    Images,
    title="Pre-BA reconstruction",
    img_kp_to_track=img_kp_to_track,
    keypoints=keypoints,
)
print(f"Total Number of 3D Points: {len(map_3d)}")
print(f"Total Number of Camera Poses: {len(camera_poses)}")

before_ba_dir = export_colmap_model(
    "colmap_exports/before_ba",
    map_3d,
    camera_poses,
    img_kp_to_track,
    keypoints,
    all_intrinsics,
    Images,
    image_dir=image_dir,
)


=== Diagnosing Map Quality ===
Total tracks in map: 5505
Tracks with negative depth in at least one observer: 15

First 10 bad tracks (tid, cam_idx, depth):
  Track 15107: depth -3.673 in camera 3
  Track 6906: depth -0.037 in camera 2
  Track 4242: depth -1.054 in camera 7
  Track 152: depth -0.318 in camera 0
  Track 951: depth -0.068 in camera 0
  Track 2343: depth -0.351 in camera 0
  Track 4607: depth -0.280 in camera 0
  Track 6227: depth -0.017 in camera 0
  Track 6296: depth -0.007 in camera 0
  Track 6383: depth -0.006 in camera 0
Total Number of 3D Points: 5505
Total Number of Camera Poses: 24
Wrote COLMAP text model to /home/gautham/Desktop/AFR/HW4/colmap_exports/before_ba (24 images, 5505 points)
Also wrote binary model (.bin) in /home/gautham/Desktop/AFR/HW4/colmap_exports/before_ba


## 6. Bundle adjustment (GTSAM)

In [8]:
try:
    optimized_poses, optimized_map_3d, ba_stats = run_bundle_adjustment(
        map_3d,
        camera_poses,
        img_kp_to_track,
        keypoints,
        all_intrinsics,
    )
    plot_3d_map(
        optimized_map_3d,
        optimized_poses,
        all_intrinsics,
        Images,
        title="Post-BA reconstruction",
        img_kp_to_track=img_kp_to_track,
        keypoints=keypoints,
    )
    after_ba_dir = export_colmap_model(
        "colmap_exports/after_ba",
        optimized_map_3d,
        optimized_poses,
        img_kp_to_track,
        keypoints,
        all_intrinsics,
        Images,
        image_dir=image_dir,
    )
except RuntimeError as e:
    print(f"BA Failed with error: {e}")
    after_ba_dir = None

Filtered 44 bad observations.
Using 5490/5505 tracks for BA.
Added 5490 landmarks with 17133 projections.
Running Levenberg-Marquardt (this should take seconds, not minutes)...
Initial error: 1.15889e+06, values: 5514
iter      cost      cost_change    lambda  success iter_time
   0   1.2697e+06     -1.1e+05      1e-05      1       0.04
iter      cost      cost_change    lambda  success iter_time
   0        1e+06      1.2e+05     0.0001      1       0.04
   1          inf            0      1e-05      0       0.03
   1          inf            0     0.0001      0       0.03
   1      5.7e+05      4.6e+05      0.001      1       0.03
   2          inf            0     0.0001      0       0.03
   2      8.8e+05     -3.1e+05      0.001      1       0.04
   2      8.7e+05       -3e+05       0.01      1       0.05
   2      4.4e+05      1.3e+05        0.1      1       0.04
   3      1.2e+06     -7.8e+05       0.01      1       0.04
   3        4e+05      3.4e+04        0.1      1       0.04


## 7. COLMAP GUI export

Models are written to `colmap_exports/before_ba` and `colmap_exports/after_ba`.
Uncomment a launch line below, or run from a terminal.

In [9]:
# open_in_colmap_gui(before_ba_dir, image_dir=image_dir)
# open_in_colmap_gui(after_ba_dir, image_dir=image_dir)

print("COLMAP models:")
print("  before BA:", "colmap_exports/before_ba")
print("  after  BA:", "colmap_exports/after_ba")
print(
    "\nLaunch:\n"
    "  colmap gui --database_path buddha_images/database.db "
    "--image_path buddha_images --import_path colmap_exports/before_ba\n"
    "  colmap gui --database_path buddha_images/database.db "
    "--image_path buddha_images --import_path colmap_exports/after_ba"
)

COLMAP models:
  before BA: colmap_exports/before_ba
  after  BA: colmap_exports/after_ba

Launch:
  colmap gui --database_path buddha_images/database.db --image_path buddha_images --import_path colmap_exports/before_ba
  colmap gui --database_path buddha_images/database.db --image_path buddha_images --import_path colmap_exports/after_ba
